# Elasticity equation with TPSA

In this tutorial we present how to solve the elasticity equation with [PyGeoN](https://github.com/compgeo-mox/pygeon) using the two-point stress approximation (TPSA) method of Nordbotten and Keilegavlen (2025), https://doi.org/10.1016/j.camwa.2025.07.035.  The unknowns are the displacement $u$, the rotation $r$, and the solid pressure $p$.

Let $\Omega=(0,1)^2$ with boundary $\partial \Omega$ and outward unit normal ${\nu}$. Given $\lambda$ the Lamé constant and $\mu$ the Kirchhoff modulus, we want to solve the following problem: find $(u, r, p)$ such that
\begin{align*}
&-\nabla \cdot \sigma(u, r, p) = b \quad \text{in } \Omega\\
&r + \mu\,{\rm asym}\,\nabla u = 0 \quad \text{in } \Omega\\
&p - \lambda\,\nabla \cdot u = 0 \quad \text{in } \Omega
\end{align*}
where the stress tensor is
$$
\sigma(u, r, p) = 2\mu\,\nabla(u) + {\rm asym}^* r + p\,I
$$
with $\epsilon$ the symmetric gradient and $b$ a body force, which is set to $0$ in this example.
For this test case we consider the footstep problem with the following boundary conditions:
$$  u = 0 \text{ on } \partial_{bottom} \Omega \qquad \nu \cdot \sigma = [0, 0]^\top \text{ on } \partial_{left} \Omega \cup \partial_{right} \Omega \qquad \nu \cdot \sigma = [0, -1e-3]^\top \text{ on } \partial_{top} \Omega $$

We present *step-by-step* how to create the grid, declare the problem data, and finally solve the problem.

First we import some of the standard modules. Since PyGeoN is based on [PorePy](https://github.com/pmgbergen/porepy) we import both modules.

In [15]:
import os

import numpy as np
import porepy as pp

import pygeon as pg

We create now the grid. TPSA works on general polygonal/polyhedral meshes; here we use a Cartesian grid for simplicity. In this example we consider a 2-dimensional structured grid, but the presented code will work also in 3d.

In [16]:
n = 10
dim = 2

sd = pp.CartGrid([n] * dim, [1] * dim)
sd = pg.convert_from_pp(sd)
sd.compute_geometry()

With the following code we set the data, in particular the Lamé and the Kirchhoff modulus, and the boundary conditions. Since we need to identify each side of $\partial \Omega$ we need a few steps.

TPSA uses the `pg.ElasticityBC` object to handle boundary conditions in a unified way. Displacement boundary conditions are set with `set_displacement_bcs` and traction conditions with `set_traction_bcs`. Any face not explicitly assigned a condition defaults to zero traction.

In [17]:
key = "elasticity_tpsa"

param = {pg.LAME_LAMBDA: 1, pg.LAME_MU: 0.5}
data = pp.initialize_data({}, key, param)

# identify the top and bottom boundaries
bottom = np.isclose(sd.face_centers[1, :], 0)
top = np.isclose(sd.face_centers[1, :], 1)
left = np.isclose(sd.face_centers[0, :], 0)
right = np.isclose(sd.face_centers[0, :], 1)

# create the BC object (also registers itself in data)
bcs = pg.ElasticityBC(sd, data, key)

# clamp the bottom: zero displacement
bcs.set_displacement_bcs(bottom)

# apply a downward traction on the top
traction = np.array([0, -1e-3])
sig_0 = np.zeros((dim, sd.num_faces))
sig_0[:, top] = traction[:, None] * sd.face_areas[top]
bcs.set_traction_bcs(top, sig_0)
bcs.set_traction_bcs(np.logical_or(left, right))

Once the data are assigned to the grid, we assemble the system matrix and the boundary right-hand side with `pg.TPSA`. The system is square with $n_{\rm dof} = (d + d(d-1)/2 + 1) \times N_c$ unknowns, where $N_c$ is the number of cells.

In [18]:
tpsa = pg.TPSA(key)

A = tpsa.assemble_system_matrix(sd, data)
rhs = tpsa.assemble_rhs_boundary_vector(sd, data)

We solve the linear system with `pg.LinearSystem` and then split the solution vector into its three components using `tpsa.split_solution`.

In [19]:
ls = pg.LinearSystem(A, rhs)
x = ls.solve()

# split into displacement, rotation, and solid pressure
u, r, p = tpsa.split_solution(sd, x)

We finally export the solution to be visualized by [ParaView](https://www.paraview.org/).

In [20]:
# reshape displacement for export: add zero z-component
cell_u = u.reshape((dim, -1))
cell_u_3d = np.vstack((cell_u, np.zeros(sd.num_cells)))

# export the solution
folder_name = os.path.join(os.getcwd(), key)
file_name = "sol"

save = pp.Exporter(sd, file_name, folder_name=folder_name)
save.write_vtu(
    [
        ("cell_u", cell_u_3d),
        ("cell_r", r),
        ("cell_p", p),
    ],
)

A representation of the computed solution is given below.

<div style="text-align: center;">
  <img src="../fig/elasticity_tpsa.png" alt="TPSA elasticity solution"/>
</div>

In [21]:
# Consistency check
assert np.isclose(np.linalg.norm(u), 0.0037743384280594424)
assert np.isclose(np.linalg.norm(r), 0.001179761302061109)
assert np.isclose(np.linalg.norm(p), 0.0035456447557315487)